# InfraScan — Improved YOLO11s Crack Detection Pipeline

This notebook implements the high-accuracy YOLO11s crack detection training pipeline on the **CRACK500** benchmark dataset.

### Architecture & Storage Design:
- **Dataset Source (Read-Only):** Google Drive at `/content/drive/MyDrive/InfraScan/processed/YOLO_CRACK500/raw_crops/`.
- **Temporary Workspace:** High-speed local NVMe storage on Colab at `/content/crack500_work/`.
- **Zero Drive Clutter:** The full dataset and training checkpoints are NOT written back to Google Drive, preserving Drive quota.
- **Model Configuration:** YOLO11s pretrained (9.4M parameters, P3–P5 feature pyramid, $640 \times 640$ resolution, AdamW optimizer).
- **Annotation Refinement:** Cohesive macro-boxes generated from segmentation masks via directional morphological closing ($11 \times 11$) and edge-distance proximity merging ($gap \le 30\text{ px}$).
- **Strict Data Validation:** 100% image-to-mask correspondence verified across train, val, and test splits prior to training launch.
- **Automated Artifact Export:** Checkpoints (`yolo_best.pt`), metric logs (`results.csv`, `evaluation_metrics.json`), and visualization plots downloaded directly via Colab download utilities.

In [1]:
# Step 1: Install Dependencies & Enforce GPU Runtime
!pip install -q "ultralytics>=8.3.0" "opencv-python-headless>=4.8" matplotlib pyyaml tqdm

import sys, torch
print('='*65)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')

if not torch.cuda.is_available():
    print('\n' + '!'*65)
    print('ERROR: No GPU detected!')
    print('Please enable GPU acceleration:')
    print('  1. In the top menu, go to: Runtime -> Change runtime type')
    print('  2. Set Hardware accelerator to: T4 GPU (or any available GPU)')
    print('  3. Click Save and re-run this notebook.')
    print('!'*65 + '\n')
    raise RuntimeError('Execution halted: CUDA GPU is required for training.')

gpu_name = torch.cuda.get_device_name(0)
gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'Active GPU      : {gpu_name} ({gpu_mem:.2f} GB VRAM)')
print('GPU verification successful! Proceeding with workspace setup.')
print('='*65)

PyTorch Version : 2.14.1+cu130
CUDA Available  : True
Active GPU      : Tesla T4 (15.64 GB VRAM)
GPU verification successful! Proceeding with workspace setup.


In [2]:
# Step 2: Mount Google Drive & Configure Source/Workspace Paths
import os, sys, shutil
from pathlib import Path

# 1. Mount Google Drive to access raw crops (Read-Only usage)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
except Exception as e:
    print('Note regarding Drive mount:', e)

# 2. Verified dataset source path in Google Drive
SOURCE_ROOT = Path('/content/drive/MyDrive/InfraScan/processed/YOLO_CRACK500/raw_crops')

# 3. Isolated temporary workspace on Colab local disk
WORK_DIR = Path('/content/crack500_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)

DATASET_DIR = WORK_DIR / 'dataset'
RUNS_DIR = WORK_DIR / 'runs'
RESULTS_DIR = WORK_DIR / 'results'

DATASET_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print('='*65)
print(f'Dataset Source (Google Drive) : {SOURCE_ROOT}')
print(f'Temporary Workspace (Local)  : {WORK_DIR}')
print('='*65)

# Verify source root exists and is not encrypted
if not SOURCE_ROOT.exists():
    raise FileNotFoundError(
        f'Source directory not found: {SOURCE_ROOT}\n'
        f'Please verify that Google Drive is mounted and that CRACK500 raw crops exist at that path.'
    )

if '.Encrypted' in str(SOURCE_ROOT):
    raise ValueError('Source path cannot be inside an encrypted directory.')

print('Source directory verified successfully!')

Mounted at /content/drive
Dataset Source (Google Drive) : /content/drive/MyDrive/InfraScan/processed/YOLO_CRACK500/raw_crops
Temporary Workspace (Local)  : /content/crack500_work
Source directory verified successfully!


In [3]:
# Step 3: High-Fidelity Macro-Box Annotation Generator
import cv2, math
import numpy as np

def generate_macro_boxes(mask, close_ksize=11, gap_threshold=30, min_size=8):
    '''
    Derives cohesive macro-boxes from crack segmentation masks:
    1. Directional morphological closing bridges micro-fractures (11x11 kernel, 2 iters).
    2. Extracts initial contour bounding boxes (filtering noise < 8px).
    3. Iteratively merges adjacent bounding boxes with edge-to-edge distance <= 30px.
    '''
    if mask is None or np.sum(mask > 0) == 0:
        return []

    # 1. Morphological closing
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (close_ksize, close_ksize))
    closed = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)

    # 2. Extract initial contours
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    boxes = []
    for c in contours:
        x, y, w, h = cv2.boundingRect(c)
        if w >= min_size and h >= min_size:
            boxes.append([x, y, x + w, y + h])

    # 3. Proximity-based merging
    merged = True
    while merged:
        merged = False
        new_boxes = []
        used = [False] * len(boxes)
        for i in range(len(boxes)):
            if used[i]: continue
            b1 = list(boxes[i])
            for j in range(i + 1, len(boxes)):
                if used[j]: continue
                b2 = boxes[j]
                dx = max(0, max(b1[0], b2[0]) - min(b1[2], b2[2]))
                dy = max(0, max(b1[1], b2[1]) - min(b1[3], b2[3]))
                dist = math.hypot(dx, dy)
                if dist <= gap_threshold:
                    b1 = [min(b1[0], b2[0]), min(b1[1], b2[1]), max(b1[2], b2[2]), max(b1[3], b2[3])]
                    used[j] = True
                    merged = True
            new_boxes.append(b1)
            used[i] = True
        boxes = new_boxes

    return boxes

In [4]:
# Step 4: Strict Image-Mask Pairing, Dataset Generation & Validation
import shutil
from tqdm.auto import tqdm

splits = ['train', 'val', 'test']
split_folder_names = {
    'train': 'traincrop',
    'val': 'valcrop',
    'test': 'testcrop'
}

stats = {}

print('='*75)
print('DISCOVERING CROPS & PREPARING YOLO DATASET WITH STRICT PAIRING')
print('='*75)

for split in splits:
    img_dir = DATASET_DIR / 'images' / split
    lbl_dir = DATASET_DIR / 'labels' / split
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    target_pattern = split_folder_names[split]

    # Recursive discovery (rglob) to handle nested directories e.g. traincrop/traincrop
    matching_dirs = [d for d in SOURCE_ROOT.rglob(target_pattern) if d.is_dir()]
    if not matching_dirs:
        raise FileNotFoundError(f"Could not locate folder matching '{target_pattern}' inside {SOURCE_ROOT}")

    # Select the directory containing the actual files
    split_dir = None
    for d in matching_dirs:
        if len(list(d.glob('*.jpg'))) > 0:
            split_dir = d
            break
    if split_dir is None:
        split_dir = sorted(matching_dirs, key=lambda p: len(p.parts), reverse=True)[0]

    print(f"\nProcessing '{split}' split from: {split_dir}")

    # Recursively discover all images and masks in split directory
    jpg_files = sorted(list(split_dir.rglob('*.jpg')))
    png_files = sorted(list(split_dir.rglob('*.png')))

    # Build strict dictionary of masks keyed by stem
    mask_map = {f.stem: f for f in png_files}

    source_count = len(jpg_files)
    matched_count = 0
    unmatched_images = []
    unmatched_masks = set(mask_map.keys())
    copied_count = 0
    labels_count = 0
    empty_labels_count = 0

    for img_path in tqdm(jpg_files, desc=f'Generating {split}'):
        stem = img_path.stem

        # 1. Verify corresponding mask exists
        if stem not in mask_map:
            unmatched_images.append(img_path.name)
            continue

        mask_path = mask_map[stem]
        matched_count += 1
        unmatched_masks.discard(stem)

        # 2. Copy image to temporary dataset directory
        dst_img = img_dir / img_path.name
        if not dst_img.exists() or dst_img.stat().st_size != img_path.stat().st_size:
            shutil.copy2(img_path, dst_img)
        copied_count += 1

        # 3. Read mask and generate macro-box annotations
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            unmatched_images.append(f"{img_path.name} (corrupt mask)")
            continue

        h, w = mask.shape[:2]
        boxes = generate_macro_boxes(mask, close_ksize=11, gap_threshold=30, min_size=8)

        # 4. Format normalized YOLO bounding boxes
        lines = []
        for b in boxes:
            pad = 4
            x1 = max(0, b[0] - pad); y1 = max(0, b[1] - pad)
            x2 = min(w, b[2] + pad); y2 = min(h, b[3] + pad)
            xc = float(np.clip(((x1 + x2) / 2.0) / w, 0.0, 1.0))
            yc = float(np.clip(((y1 + y2) / 2.0) / h, 0.0, 1.0))
            bw = float(np.clip((x2 - x1) / w, 0.0, 1.0))
            bh = float(np.clip((y2 - y1) / h, 0.0, 1.0))
            if bw > 0 and bh > 0:
                lines.append(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

        # Write label file
        dst_lbl = lbl_dir / f"{stem}.txt"
        with open(dst_lbl, 'w') as f:
            f.writelines(lines)

        labels_count += 1
        if len(lines) == 0:
            empty_labels_count += 1

    stats[split] = {
        'source_images': source_count,
        'source_masks': len(png_files),
        'matched_masks': matched_count,
        'images_copied': copied_count,
        'labels_generated': labels_count,
        'unmatched_images': len(unmatched_images),
        'unmatched_masks': len(unmatched_masks),
        'empty_labels': empty_labels_count
    }

print('\n' + '='*75)
print('STRICT DATASET INTEGRITY & VALIDATION REPORT')
print('='*75)
header = f"{'Split':<8} | {'Images':<8} | {'Masks':<8} | {'Matched':<8} | {'Copied':<8} | {'Labels':<8} | {'Unmatched':<10} | {'Empty':<6}"
print(header)
print('-'*75)

all_valid = True
for split, s in stats.items():
    row = f"{split:<8} | {s['source_images']:<8} | {s['source_masks']:<8} | {s['matched_masks']:<8} | {s['images_copied']:<8} | {s['labels_generated']:<8} | {s['unmatched_images']:<10} | {s['empty_labels']:<6}"
    print(row)

    # Strict assertions
    if s['source_images'] == 0:
        print(f"ERROR: Split '{split}' has 0 source images!")
        all_valid = False
    if s['matched_masks'] != s['source_images']:
        print(f"ERROR: Split '{split}' has mask mismatches ({s['matched_masks']} != {s['source_images']})!")
        all_valid = False
    if s['images_copied'] != s['source_images']:
        print(f"ERROR: Split '{split}' failed image copy ({s['images_copied']} != {s['source_images']})!")
        all_valid = False
    if s['labels_generated'] != s['source_images']:
        print(f"ERROR: Split '{split}' failed label generation ({s['labels_generated']} != {s['source_images']})!")
        all_valid = False
    if s['unmatched_images'] > 0:
        print(f"ERROR: Split '{split}' has {s['unmatched_images']} unmatched images!")
        all_valid = False

print('='*75)

if not all_valid:
    raise RuntimeError('Dataset validation failed! Training aborted to prevent training on corrupt/incomplete data.')

print('All splits verified with 100% image-mask correspondence! Dataset is fully prepared.')

DISCOVERING CROPS & PREPARING YOLO DATASET WITH STRICT PAIRING

Processing 'train' split from: /content/drive/MyDrive/InfraScan/processed/YOLO_CRACK500/raw_crops/traincrop/traincrop


Generating train:   0%|          | 0/1896 [00:00<?, ?it/s]


Processing 'val' split from: /content/drive/MyDrive/InfraScan/processed/YOLO_CRACK500/raw_crops/valcrop/valcrop


Generating val:   0%|          | 0/348 [00:00<?, ?it/s]


Processing 'test' split from: /content/drive/MyDrive/InfraScan/processed/YOLO_CRACK500/raw_crops/testcrop/testcrop


Generating test:   0%|          | 0/1124 [00:00<?, ?it/s]


STRICT DATASET INTEGRITY & VALIDATION REPORT
Split    | Images   | Masks    | Matched  | Copied   | Labels   | Unmatched  | Empty 
---------------------------------------------------------------------------
train    | 1896     | 1896     | 1896     | 1896     | 1896     | 0          | 0     
val      | 348      | 348      | 348      | 348      | 348      | 0          | 0     
test     | 1124     | 1124     | 1124     | 1124     | 1124     | 0          | 0     
All splits verified with 100% image-mask correspondence! Dataset is fully prepared.


In [5]:
# Step 5: Configure and Validate data.yaml
import yaml

data_yaml_path = DATASET_DIR / 'data.yaml'
data_config = {
    'path': str(DATASET_DIR.resolve()),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'crack'}
}

with open(data_yaml_path, 'w') as f:
    yaml.dump(data_config, f, sort_keys=False)

print(f'data.yaml generated at: {data_yaml_path}')
print('Contents:')
with open(data_yaml_path, 'r') as f:
    print(f.read())

# Verify physical path existence
assert (DATASET_DIR / 'images' / 'train').exists(), 'Missing images/train!'
assert (DATASET_DIR / 'images' / 'val').exists(), 'Missing images/val!'
assert (DATASET_DIR / 'images' / 'test').exists(), 'Missing images/test!'
assert (DATASET_DIR / 'labels' / 'train').exists(), 'Missing labels/train!'
assert (DATASET_DIR / 'labels' / 'val').exists(), 'Missing labels/val!'
assert (DATASET_DIR / 'labels' / 'test').exists(), 'Missing labels/test!'

train_imgs = len(list((DATASET_DIR / 'images' / 'train').glob('*')))
val_imgs = len(list((DATASET_DIR / 'images' / 'val').glob('*')))
test_imgs = len(list((DATASET_DIR / 'images' / 'test').glob('*')))

print(f'Verification: train images={train_imgs}, labels={len(list((DATASET_DIR / "labels" / "train").glob("*")))}')
print(f'Verification: val images={val_imgs}, labels={len(list((DATASET_DIR / "labels" / "val").glob("*")))}')
print(f'Verification: test images={test_imgs}, labels={len(list((DATASET_DIR / "labels" / "test").glob("*")))}')

assert train_imgs > 0, 'images/train is empty!'
assert val_imgs > 0, 'images/val is empty!'
assert test_imgs > 0, 'images/test is empty!'

print('data.yaml paths and physical directories confirmed intact and non-empty!')

data.yaml generated at: /content/crack500_work/dataset/data.yaml
Contents:
path: /content/crack500_work/dataset
train: images/train
val: images/val
test: images/test
names:
  0: crack

Verification: train images=1896, labels=1896
Verification: val images=348, labels=348
Verification: test images=1124, labels=1124
data.yaml paths and physical directories confirmed intact and non-empty!


In [6]:
# Step 6: Train YOLO11s in Temporary Storage (Preserving Hyperparameters)
import time, torch
from ultralytics import YOLO

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is unavailable! Training cannot proceed without GPU.')

print('='*65)
print('INITIALIZING YOLO11s (P3-P5 Architecture, 9.4M Parameters)...')
print('='*65)
model = YOLO('yolo11s.pt')

t0 = time.time()
train_results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    patience=10,
    save=True,
    plots=True,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    box=8.5,            # Elevated boundary regression weight
    dfl=2.0,            # Distribution focal loss for thin edge detail
    cls=0.5,            # Single class weight
    degrees=15.0,       # Rotation augmentation for arbitrary crack orientations
    mosaic=0.5,         # Controlled mosaic to maintain linear crack continuity
    close_mosaic=10,    # Disable mosaic during last 10 epochs for fine localization
    fliplr=0.5,
    flipud=0.5,
    scale=0.5,
    project=str(RUNS_DIR),
    name='crack500_yolo11s_improved',
    exist_ok=True
)
training_duration = time.time() - t0
print(f'\nTraining completed in {training_duration / 60:.2f} minutes.')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
INITIALIZING YOLO11s (P3-P5 Architecture, 9.4M Parameters)...
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=8.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/crack500_work/dataset/data.yaml, degrees=15.0, deterministic=True, device=, dfl=2.0, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, e

In [9]:

# Step 7: Strict Evaluation on Held-Out Test Set & Results Compilation

import json
import time
from pathlib import Path
from ultralytics import YOLO

# --------------------------------------------------
# 1. Validate paths and load the best trained model
# --------------------------------------------------

train_run_dir = RUNS_DIR / "crack500_yolo11s_improved"
best_weights_path = train_run_dir / "weights" / "best.pt"

if not best_weights_path.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found at: {best_weights_path}"
    )

# Ensure output directories exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)

eval_model = YOLO(str(best_weights_path))

# --------------------------------------------------
# 2. Evaluate on the held-out test split
# --------------------------------------------------

print("=" * 70)
print("EVALUATING BEST MODEL ON HELD-OUT TEST SPLIT (split=test)")
print("=" * 70)

t_eval_start = time.time()

test_results = eval_model.val(
    data=str(data_yaml_path),
    split="test",
    imgsz=640,
    batch=16,
    plots=True,
    project=str(RUNS_DIR),
    name="test_evaluation",
    exist_ok=True
)

eval_duration = time.time() - t_eval_start

# --------------------------------------------------
# 3. Extract evaluation metrics
# --------------------------------------------------

# mp and mr return mean precision and recall across classes
p = float(test_results.box.mp)
r = float(test_results.box.mr)
map50 = float(test_results.box.map50)
map50_95 = float(test_results.box.map)

# Ultralytics reports inference time in milliseconds per image
speed_ms = float(test_results.speed["inference"])

# Validate extracted values
metric_values = {
    "precision": p,
    "recall": r,
    "mAP50": map50,
    "mAP50_95": map50_95,
    "inference_speed_ms": speed_ms
}

for metric_name, metric_value in metric_values.items():
    if not isinstance(metric_value, (int, float)) or not (
        metric_value >= 0
    ):
        raise ValueError(
            f"Invalid evaluation metric: {metric_name} = {metric_value}"
        )

# --------------------------------------------------
# 4. Benchmark comparison
# --------------------------------------------------

baseline = {
    "precision": 0.597,
    "recall": 0.380,
    "mAP50": 0.400,
    "mAP50_95": 0.221
}

print("\n" + "=" * 70)
print("          INFRASCAN YOLO PERFORMANCE BENCHMARK")
print("=" * 70)

print(
    f"{'Metric':<18} | "
    f"{'Milestone 2 (YOLO11n)':<22} | "
    f"{'Improved (YOLO11s)':<20}"
)
print("-" * 70)

print(f"{'Precision':<18} | {baseline['precision']:<22.4f} | {p:<20.4f}")
print(f"{'Recall':<18} | {baseline['recall']:<22.4f} | {r:<20.4f}")
print(f"{'mAP@50':<18} | {baseline['mAP50']:<22.4f} | {map50:<20.4f}")
print(f"{'mAP@50-95':<18} | {baseline['mAP50_95']:<22.4f} | {map50_95:<20.4f}")
print(f"{'Inference Speed':<18} | {'N/A':<22} | {speed_ms:<20.2f} ms")

print("=" * 70)

# --------------------------------------------------
# 5. Calculate metric improvements
# --------------------------------------------------

improvements = {
    "precision_gain": round(p - baseline["precision"], 4),
    "recall_gain": round(r - baseline["recall"], 4),
    "mAP50_gain": round(map50 - baseline["mAP50"], 4),
    "mAP50_95_gain": round(map50_95 - baseline["mAP50_95"], 4)
}

print("\nMetric improvements over Milestone 2:")
for metric_name, gain in improvements.items():
    print(f"{metric_name:<20}: {gain:+.4f}")

# --------------------------------------------------
# 6. Export evaluation metrics to JSON
# --------------------------------------------------

metrics_data = {
    "model": "YOLO11s",
    "architecture": "P3-P5",
    "task": "crack_detection",
    "dataset": "CRACK500",
    "image_size": 640,

    "training_duration_seconds": round(training_duration, 2),
    "test_evaluation_duration_seconds": round(eval_duration, 2),

    "test_split": {
        "precision": round(p, 4),
        "recall": round(r, 4),
        "mAP50": round(map50, 4),
        "mAP50_95": round(map50_95, 4),
        "inference_speed_ms_per_image": round(speed_ms, 2)
    },

    "baseline_comparison": {
        "baseline_model": "YOLO11n",
        "baseline_metrics": baseline,
        "improvements": improvements
    },

    "annotation_note": (
        "Milestone 2 evaluated on fragmented connected-component "
        "pseudo-boxes. Improved YOLO11s evaluates on cohesive "
        "macro-crack annotations derived using morphological closing "
        "(11x11 kernel) and proximity merging (<=30 pixels). "
        "The annotation-generation approaches differ, so the benchmark "
        "comparison should be interpreted with this limitation."
    )
}

metrics_json_path = RESULTS_DIR / "evaluation_metrics.json"

with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics_data, f, indent=4)

print(f"\nEvaluation metrics saved to: {metrics_json_path}")

# --------------------------------------------------
# 7. Generate training and evaluation summary
# --------------------------------------------------

summary_path = RESULTS_DIR / "training_summary.txt"

with open(summary_path, "w", encoding="utf-8") as f:

    f.write("INFRASCAN YOLO TRAINING & EVALUATION REPORT\n")
    f.write("=" * 60 + "\n\n")

    f.write("MODEL CONFIGURATION\n")
    f.write("-" * 60 + "\n")
    f.write("Model Architecture : YOLO11s (P3-P5)\n")
    f.write("Dataset            : CRACK500\n")
    f.write("Task               : Crack Detection\n")
    f.write("Image Resolution   : 640x640\n")
    f.write(f"Training Duration  : {training_duration / 60:.2f} minutes\n")
    f.write(f"Test Evaluation    : {eval_duration:.2f} seconds\n\n")

    f.write("HELD-OUT TEST RESULTS\n")
    f.write("-" * 60 + "\n")
    f.write(f"Precision          : {p:.4f} (Baseline: 0.597)\n")
    f.write(f"Recall             : {r:.4f} (Baseline: 0.380)\n")
    f.write(f"mAP@50             : {map50:.4f} (Baseline: 0.400)\n")
    f.write(f"mAP@50-95          : {map50_95:.4f} (Baseline: 0.221)\n")
    f.write(f"Inference Latency  : {speed_ms:.2f} ms/image\n\n")

    f.write("IMPROVEMENTS OVER MILESTONE 2\n")
    f.write("-" * 60 + "\n")
    f.write(f"Precision Gain     : {improvements['precision_gain']:+.4f}\n")
    f.write(f"Recall Gain        : {improvements['recall_gain']:+.4f}\n")
    f.write(f"mAP@50 Gain        : {improvements['mAP50_gain']:+.4f}\n")
    f.write(f"mAP@50-95 Gain     : {improvements['mAP50_95_gain']:+.4f}\n\n")

    f.write("ANNOTATION LIMITATION\n")
    f.write("-" * 60 + "\n")
    f.write(metrics_data["annotation_note"] + "\n\n")

    f.write("=" * 60 + "\n")
    f.write("END OF REPORT\n")

print(f"Training summary saved to: {summary_path}")

# --------------------------------------------------
# 8. Final output locations
# --------------------------------------------------

print("\n" + "=" * 70)
print("EVALUATION AND RESULTS COMPILATION COMPLETED")
print("=" * 70)

print(f"Best model checkpoint : {best_weights_path}")
print(f"Evaluation plots      : {RUNS_DIR / 'test_evaluation'}")
print(f"Metrics JSON          : {metrics_json_path}")
print(f"Training summary      : {summary_path}")
print(f"Evaluation duration   : {eval_duration:.2f} seconds")


EVALUATING BEST MODEL ON HELD-OUT TEST SPLIT (split=test)
Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO11s summary (fused): 100 layers, 9,413,187 parameters, 0 gradients, 21.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1166.1±431.5 MB/s, size: 61.4 KB)
val: Scanning /content/crack500_work/dataset/labels/test.cache... 1124 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1124/1124 248.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 71/71 5.3it/s 13.5s
                   all       1124       1376      0.845      0.748      0.806      0.517
Speed: 0.8ms preprocess, 5.7ms inference, 0.0ms loss, 1.4ms postprocess per image
Results saved to /content/crack500_work/runs/test_evaluation

          INFRASCAN YOLO PERFORMANCE BENCHMARK
Metric             | Milestone 2 (YOLO11n)  | Improved (YOLO11s)  
------------------------------------------------------------------

In [10]:
# Step 8: Verify Output Artifacts & Trigger Automated Downloads
import os, shutil, zipfile
from pathlib import Path
from google.colab import files

train_run_dir = RUNS_DIR / 'crack500_yolo11s_improved'
test_eval_dir = RUNS_DIR / 'test_evaluation'

print('='*65)
print('COLLECTING & VERIFYING RESULTS ARTIFACTS...')
print('='*65)

# 1. Best model checkpoint
best_pt_src = train_run_dir / 'weights' / 'best.pt'
best_pt_dst = RESULTS_DIR / 'yolo_best.pt'
if best_pt_src.exists():
    shutil.copy2(best_pt_src, best_pt_dst)
    print(f'[OK] Checkpoint verified: {best_pt_dst} ({best_pt_dst.stat().st_size / 1e6:.2f} MB)')
else:
    raise FileNotFoundError(f'Missing checkpoint: {best_pt_src}')

# 2. Training metrics CSV
results_csv_src = train_run_dir / 'results.csv'
results_csv_dst = RESULTS_DIR / 'results.csv'
if results_csv_src.exists():
    shutil.copy2(results_csv_src, results_csv_dst)
    print(f'[OK] Results CSV verified: {results_csv_dst}')
else:
    print(f'[Warning] results.csv not found at {results_csv_src}')

# 3. Collect confusion matrix, PR curves, and prediction plots
plot_patterns = ['*.png', '*.jpg']
collected_plots = []
for p_dir in [train_run_dir, test_eval_dir]:
    if p_dir.exists():
        for pat in plot_patterns:
            for img_p in p_dir.glob(pat):
                target = RESULTS_DIR / f'{p_dir.name}_{img_p.name}'
                shutil.copy2(img_p, target)
                collected_plots.append(target)

print(f'[OK] Collected {len(collected_plots)} plot images into {RESULTS_DIR}')

# 4. Bundle all metrics, plots, and summary into a zip archive
artifacts_zip_path = RESULTS_DIR / 'yolo_training_artifacts.zip'
with zipfile.ZipFile(artifacts_zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for item in RESULTS_DIR.iterdir():
        if item.name != 'yolo_best.pt' and item.name != 'yolo_training_artifacts.zip':
            z.write(item, arcname=item.name)

print(f'[OK] Artifacts zip archive created: {artifacts_zip_path} ({artifacts_zip_path.stat().st_size / 1e3:.1f} KB)')

# 5. Check existence of all required items before downloading
required_artifacts = [
    RESULTS_DIR / 'yolo_best.pt',
    RESULTS_DIR / 'results.csv',
    RESULTS_DIR / 'evaluation_metrics.json',
    RESULTS_DIR / 'training_summary.txt',
    artifacts_zip_path
]

print('\n' + '='*65)
print('VERIFICATION OF ARTIFACTS PRIOR TO DOWNLOAD:')
print('='*65)
all_exist = True
for req in required_artifacts:
    status = 'EXISTS' if req.exists() else 'MISSING'
    print(f'{req.name:<32} : {status}')
    if not req.exists():
        all_exist = False

if not all_exist:
    raise RuntimeError('One or more required artifacts are missing! Halting before download.')

print('='*65)
print('All required output files exist! Triggering browser downloads...')
print('='*65)

# 6. Trigger downloads directly to the user computer
print('1/4 Downloading trained weights (yolo_best.pt)...')
files.download(str(RESULTS_DIR / 'yolo_best.pt'))

print('2/4 Downloading training artifacts bundle (plots, confusion matrix, CSV, JSON)...')
files.download(str(artifacts_zip_path))

print('3/4 Downloading evaluation_metrics.json...')
files.download(str(RESULTS_DIR / 'evaluation_metrics.json'))

print('4/4 Downloading results.csv...')
files.download(str(RESULTS_DIR / 'results.csv'))

print('\nAll downloads initiated successfully!')
print('\n' + '!'*65)
print('IMPORTANT NOTE ON TEMPORARY STORAGE:')
print('All dataset files and training outputs under /content/crack500_work/ are')
print('temporary and will disappear when the Colab runtime disconnects or restarts.')
print('Please ensure yolo_best.pt is saved locally to:')
print('  E:\\infosys\\Internship\\InfraScan\\models\\yolo_best.pt')
print('!'*65)

COLLECTING & VERIFYING RESULTS ARTIFACTS...
[OK] Checkpoint verified: /content/crack500_work/results/yolo_best.pt (19.18 MB)
[OK] Results CSV verified: /content/crack500_work/results/results.csv
[OK] Collected 32 plot images into /content/crack500_work/results
[OK] Artifacts zip archive created: /content/crack500_work/results/yolo_training_artifacts.zip (12699.2 KB)

VERIFICATION OF ARTIFACTS PRIOR TO DOWNLOAD:
yolo_best.pt                     : EXISTS
results.csv                      : EXISTS
evaluation_metrics.json          : EXISTS
training_summary.txt             : EXISTS
yolo_training_artifacts.zip      : EXISTS
All required output files exist! Triggering browser downloads...
1/4 Downloading trained weights (yolo_best.pt)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

2/4 Downloading training artifacts bundle (plots, confusion matrix, CSV, JSON)...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

... (3 outputs left out)

4/4 Downloading results.csv...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All downloads initiated successfully!

!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
IMPORTANT NOTE ON TEMPORARY STORAGE:
All dataset files and training outputs under /content/crack500_work/ are
temporary and will disappear when the Colab runtime disconnects or restarts.
Please ensure yolo_best.pt is saved locally to:
  E:\infosys\Internship\InfraScan\models\yolo_best.pt
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
